# PageRank: Teleportation & the Google Matrix

## Learning Objectives

1. **Define** the teleportation fix and its effect on spider traps and dead ends
2. **Construct** the Google matrix $G = \beta M + (1-\beta)/n \cdot \mathbf{1}\mathbf{1}^T$
3. **Implement** PageRank with teleportation
4. **Explain** how to handle dangling nodes (dead ends) at scale

## Teleportation Fix

At each step, the surfer:
- With probability $\beta$ (damping factor): follows a random outgoing link
- With probability $(1-\beta)$: jumps to a **uniformly random** page

Typical value: $\beta = 0.85$ (15% teleportation).

**Effect:**
- **Spider traps:** surfer escapes with probability $1-\beta$ at each step
- **Dead ends:** surfer always teleports (no outgoing links to follow)
- **Convergence:** guaranteed because the Google matrix is irreducible and aperiodic

## The Google Matrix

$$G = \beta M + \frac{1-\beta}{n} \mathbf{1}\mathbf{1}^T$$

- $M$: column-stochastic transition matrix (with dead ends handled)
- $\mathbf{1}\mathbf{1}^T$: $n \times n$ all-ones matrix divided by $n$ = uniform jump

**PageRank update:**
$$\mathbf{r}^{(t+1)} = \beta M \mathbf{r}^{(t)} + \frac{1-\beta}{n} \mathbf{1}$$

This avoids explicitly storing the $n \times n$ Google matrix — only $M$ is needed, which is sparse.

In [ ]:
from collections import defaultdict

def pagerank_teleportation(out_edges, nodes, beta=0.85, n_iter=100):
    """PageRank with teleportation fix for spider traps and dead ends."""
    n = len(nodes); out_deg = {u: len(out_edges.get(u,[])) for u in nodes}
    r = {v: 1.0/n for v in nodes}
    for _ in range(n_iter):
        new_r = {v: 0.0 for v in nodes}
        leak = 0.0  # rank from dead ends
        for u in nodes:
            if out_deg[u] > 0:
                for v in out_edges.get(u,[]):
                    new_r[v] += beta * r[u] / out_deg[u]
            else:
                leak += r[u]  # dead end: redistribute uniformly
        # Add teleportation and leaked rank
        teleport = (1 - beta + beta * leak) / n
        for v in nodes:
            new_r[v] += teleport
        r = new_r
    return r

# Test on spider trap graph
nodes = ['A','B','C','D']
out_sp = {'A':['B'],'B':['C'],'C':['B'],'D':['A']}
r_sp = pagerank_teleportation(out_sp, nodes, beta=0.85)
print("Spider trap graph WITH teleportation:")
for v,rank in r_sp.items():
    bar = '#'*int(rank*40)
    print(f"  {v}: {rank:.4f}  {bar}")
print(f"Total: {sum(r_sp.values()):.4f}")

In [ ]:
# Test on dead end graph
out_dead = {'A':['B','C'],'B':['C']}  # C is dead end
r_dead = pagerank_teleportation(out_dead, nodes[:3], beta=0.85)
print("Dead end graph WITH teleportation:")
for v,rank in r_dead.items():
    bar = '#'*int(rank*40)
    print(f"  {v}: {rank:.4f}  {bar}")
print(f"Total: {sum(r_dead.values()):.4f}")

# Compare effect of beta
print("
Effect of beta (damping factor) on healthy graph:")
nodes_h = ['A','B','C','D']
out_h = {'A':['B','C'],'B':['D'],'C':['B','D'],'D':['A']}
for beta in [0.5, 0.7, 0.85, 0.95]:
    r = pagerank_teleportation(out_h, nodes_h, beta=beta)
    print(f"  beta={beta:.2f}: A={r['A']:.4f} B={r['B']:.4f} C={r['C']:.4f} D={r['D']:.4f}")

## Dangling Nodes at Scale

For web-scale PageRank ($n \sim 10^{12}$), the teleportation vector $(1-\beta)/n$ is tiny but present.

**Dangling node handling:** rather than tracking all dead ends explicitly, redistribute their rank:

$$\mathbf{r}^{(t+1)} = \beta M \mathbf{r}^{(t)} + \frac{\beta \cdot \text{dangling\_sum} + (1-\beta)}{n} \mathbf{1}$$

where $\text{dangling\_sum} = \sum_{u:\text{dead end}} r^{(t)}_u$.

**Memory:** only the rank vector ($n$ floats) and sparse matrix $M$ are needed.
At $n = 10^{12}$ pages with 8-byte floats: ~8 TB for ranks alone — distributed across many machines.

## Summary

| Issue | Without teleportation | With teleportation ($\beta=0.85$) |
|-------|----------------------|----------------------------------|
| Spider traps | Rank absorbed | Surfer escapes with prob 0.15 |
| Dead ends | Rank leaks | Redistributed uniformly |
| Convergence | May fail | Always converges |
| Total rank | May < 1 | Always = 1 |

The teleportation parameter $\beta = 0.85$ is the original Google choice.
Lower $\beta$ → faster convergence but more uniform ranks (less discriminative).
Higher $\beta$ → more discriminative but slower convergence and more trap sensitivity.